# 02 — Inversión magnetométrica 3D independiente

Inversión TMI suave sobre **exactamente la misma malla** de gravimetría. Se reserva 20 % de estaciones para validación espacial ciega.

In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from mantle3d.magnetics import load_config,load_magnetics,run_inversion,save_result
config=load_config(ROOT); magnetic_data=load_magnetics(ROOT,config)
magnetic_data.describe().round(3)

## Datos y separación de validación

In [ ]:
xyz=magnetic_data[['X','Y','Z']].to_numpy(); values=magnetic_data['Anomalia (nT)'].to_numpy()
order=np.lexsort((xyz[:,1],xyz[:,0])); hold=order[::config['validation']['holdout_stride']]; train=np.setdiff1d(np.arange(len(values)),hold)
fig,ax=plt.subplots(figsize=(8,6)); sc=ax.scatter(xyz[train,0],xyz[train,1],c=values[train],cmap='RdBu_r',s=35,label='Inversión'); ax.scatter(xyz[hold,0],xyz[hold,1],marker='s',facecolors='none',edgecolors='k',s=55,label='Validación'); ax.set(title='Anomalía magnética y estaciones retenidas',xlabel='X / Este (m)',ylabel='Y / Norte (m)',aspect='equal'); ax.ticklabel_format(style='plain'); ax.legend(); fig.colorbar(sc,ax=ax,label='Anomalía TMI (nT)'); plt.show()

## Campo inductor y significado del modelo

Se usa IGRF-14 evaluado provisionalmente el 2026-07-01 en el centro del área. SimPEG interpreta X como Este, Y como Norte, Z positivo arriba y la declinación horaria desde el norte, compatible con EPSG:9377. El parámetro invertido es **susceptibilidad magnética absoluta/efectiva positiva** en SI, bajo la hipótesis de magnetización inducida: $\chi \ge 0$. No se interpreta como contraste firmado respecto a un background. El background y modelo de referencia L2 son $\chi=0$; el modelo inicial es el valor pequeño, no negativo e interior indicado en la configuración. Los límites se imponen durante la optimización ProjectedGNCG, sin truncamiento posterior. La fecha del campo debe reemplazarse por la fecha real de adquisición.

In [ ]:
pd.Series(config['inducing_field'],name='valor').to_frame()

## Malla común e inversión

La malla tensorial común usa la topografía del DEM `data/processed/topography/topography_xyz_corregida.txt`, complementada con elevaciones de estaciones, y se carga desde `outputs/common_mesh`; no se reconstruye. Se mantienen la regularización L2, los pesos de sensibilidad y el enfriamiento de beta. La susceptibilidad se restringe dentro de ProjectedGNCG a `lower = 0.0` y `upper = 0.2` SI.

In [ ]:
result=run_inversion(ROOT); mesh=result['mesh']
print(f'Malla: {mesh.shape_cells}; celdas={mesh.n_cells:,}; activas={result["active"].sum():,}')
print(f'Nodos X: {mesh.nodes_x.min():.3f} a {mesh.nodes_x.max():.3f} m')
print(f'Nodos Y: {mesh.nodes_y.min():.3f} a {mesh.nodes_y.max():.3f} m')
print(f'Centros X: {mesh.cell_centers_x.min():.3f} a {mesh.cell_centers_x.max():.3f} m')
print(f'Centros Y: {mesh.cell_centers_y.min():.3f} a {mesh.cell_centers_y.max():.3f} m')
print(f'SHA-256: {result["mesh_manifest"]["mesh_sha256"]}')
print(f'Topografía: {result["mesh_manifest"].get("topography_path", result["mesh_manifest"].get("topography_source"))}')
print(f'Incertidumbre: {result["uncertainty_floor"]:.2f} nT; modelo: {result["model_active"].min():.4f} a {result["model_active"].max():.4f} SI')

## Malla en el dominio del modelo

In [ ]:
topo=result['topography']; fig,axes=plt.subplots(1,2,figsize=(15,6),constrained_layout=True)
for x in mesh.nodes_x: axes[0].plot([x,x],[mesh.nodes_y.min(),mesh.nodes_y.max()],color='.78',lw=.35)
for y in mesh.nodes_y: axes[0].plot([mesh.nodes_x.min(),mesh.nodes_x.max()],[y,y],color='.78',lw=.35)
axes[0].scatter(xyz[:,0],xyz[:,1],s=12,c='#E76F51',zorder=3); axes[0].set(title='Vista en planta',xlabel='X (m)',ylabel='Y (m)',aspect='equal'); axes[0].ticklabel_format(style='plain')
for x in mesh.nodes_x: axes[1].plot([x,x],[mesh.nodes_z.min(),mesh.nodes_z.max()],color='.8',lw=.35)
for z in mesh.nodes_z: axes[1].plot([mesh.nodes_x.min(),mesh.nodes_x.max()],[z,z],color='.8',lw=.35)
y_slice=mesh.cell_centers_y[mesh.shape_cells[1]//2]; y0=np.unique(topo[:,1])[np.argmin(abs(np.unique(topo[:,1])-y_slice))]; p=topo[np.isclose(topo[:,1],y0)]; p=p[np.argsort(p[:,0])]; axes[1].plot(p[:,0],p[:,2],color='#2A9D8F',lw=2); axes[1].scatter(xyz[:,0],xyz[:,2],s=10,c='#E76F51'); axes[1].set_xlim(mesh.nodes_x.min(),mesh.nodes_x.max()); axes[1].set(title=f'Sección X–Z central Y={y0:.0f} m',xlabel='X (m)',ylabel='Elevación (m)'); axes[1].ticklabel_format(style='plain'); plt.show()

## Ajuste y validación ciega

In [ ]:
train=result['train']; hold=result['holdout']; obs=result['observed']; pt=result['predicted_train']; ph=result['predicted_holdout']
rmse_t=np.sqrt(np.mean((pt-obs[train])**2)); rmse_h=np.sqrt(np.mean((ph-obs[hold])**2)); corr_t=np.corrcoef(pt,obs[train])[0,1]; corr_h=np.corrcoef(ph,obs[hold])[0,1]
display(pd.DataFrame({'conjunto':['Inversión','Validación ciega'],'n':[len(train),len(hold)],'RMSE_nT':[rmse_t,rmse_h],'correlación':[corr_t,corr_h]}).round(3))
fig,ax=plt.subplots(figsize=(7,6),constrained_layout=True)
all_obs=np.r_[obs[train],obs[hold]]; all_pred=np.r_[pt,ph]; lim=[min(all_obs.min(),all_pred.min()),max(all_obs.max(),all_pred.max())]
ax.scatter(obs[train],pt,c='#2A9D8F',edgecolor='white',s=42,label='Inversión')
ax.scatter(obs[hold],ph,marker='s',facecolors='none',edgecolors='#C44536',s=64,linewidths=1.5,label='Validación ciega')
ax.plot(lim,lim,'k--',lw=1,label='1:1')
ax.set(title='Predicho vs observado',xlabel='Observado (nT)',ylabel='Predicho (nT)')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal',adjustable='box'); ax.grid(True,alpha=.25); ax.legend()
fig_dir=ROOT/'outputs/figures'; fig_dir.mkdir(parents=True,exist_ok=True)
fig.savefig(fig_dir/'magnetics_predicho_vs_observado.png',dpi=300)
fig.savefig(fig_dir/'magnetics_predicho_vs_observado.svg')
plt.show()

## Diagnóstico de convergencia

In [ ]:
h=result['history']; it=np.arange(1,len(h['phi_d'])+1); r=(pt-obs[train])/result['uncertainty_floor']; target=len(train)*config['inversion']['target_chi_factor']; lo=config['inversion']['effective_susceptibility_lower_si']; hi=config['inversion']['effective_susceptibility_upper_si']; m=result['model_active']; tol=1e-12
raw_negative_pct=np.mean(m<0.0)*100; physical_negative_pct=np.mean(m < -tol)*100; at_lower=np.mean(np.isclose(m,lo,atol=tol,rtol=0))*100; at_upper=np.mean(np.isclose(m,hi,atol=tol,rtol=0))*100
print(f'model_active.min() >= 0: {m.min() >= 0.0}')
print(f'Celdas negativas: {raw_negative_pct:.6f} %; negativas físicas (< -{tol:g}): {physical_negative_pct:.6f} %')
print(f'Celdas en lower bound: {at_lower:.3f} %; en upper bound: {at_upper:.3f} %')
assert m.min() >= -tol, 'La inversión produjo susceptibilidad negativa fuera de tolerancia numérica'
display(pd.DataFrame({'métrica':['Iteraciones','phi_d inicial','phi_d final','phi_d / n_d','Objetivo phi_d','Residuo normalizado medio','Desv. residuo normalizado','|residuo| ≤ 2σ','Mínimo susceptibilidad (SI)','Máximo susceptibilidad (SI)','Celdas negativas físicas (%)','Celdas en lower bound (%)','Celdas en upper bound (%)','RMSE validación','Correlación validación'],'valor':[len(it),h['phi_d'][0],h['phi_d'][-1],h['phi_d'][-1]/len(train),target,r.mean(),r.std(),np.mean(abs(r)<=2)*100,m.min(),m.max(),physical_negative_pct,at_lower,at_upper,rmse_h,corr_h]}).round(6))
print('CONVERGIÓ al objetivo estadístico.' if h['phi_d'][-1]<=target else 'NO alcanzó el objetivo estadístico.')
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True); axes[0].semilogy(it,h['phi_d'],'o-'); axes[0].axhline(target,color='k',ls='--'); axes[0].set(xlabel='Iteración',ylabel='Misfit',title='Convergencia del ajuste'); axes[1].semilogy(it,h['beta'],'o-',color='#7161A8'); axes[1].set(xlabel='Iteración',ylabel='Beta',title='Enfriamiento de regularización'); plt.show()

## Secciones del modelo recuperado

In [ ]:
model=result['model_full']; vmax=float(np.nanmax(model)); active_z=mesh.cell_centers[result['active'],2]; z_ind=int(np.argmin(np.abs(mesh.cell_centers_z-np.nanmax(active_z)))); y_ind=mesh.shape_cells[1]//2; y_slice=mesh.cell_centers_y[y_ind]; y_levels=np.unique(result['topography'][:,1]); y0=y_levels[np.argmin(abs(y_levels-y_slice))]; p=result['topography'][np.isclose(result['topography'][:,1],y0)]; p=p[np.argsort(p[:,0])]; zmin=float(np.nanmin(active_z)); zmax=float(max(np.nanmax(result['topography'][:,2]),np.nanmax(active_z))); fig=plt.figure(figsize=(15,5.5),constrained_layout=True); gs=fig.add_gridspec(1,3,width_ratios=[1,1,.045]); axes=[fig.add_subplot(gs[0,0]),fig.add_subplot(gs[0,1])]; cax=fig.add_subplot(gs[0,2]); opts={'cmap':'viridis','vmin':0.0,'vmax':vmax}; mesh.plot_slice(model,normal='Z',ind=z_ind,ax=axes[0],grid=False,pcolor_opts=opts); axes[0].set_title(f'Sección horizontal z={mesh.cell_centers_z[z_ind]:.0f} m'); mesh.plot_slice(model,normal='Y',ind=y_ind,ax=axes[1],grid=False,pcolor_opts=opts); axes[1].plot(p[:,0],p[:,2],color='k',lw=1.5,label='Topografía'); axes[1].set_xlim(mesh.nodes_x.min(),mesh.nodes_x.max()); axes[1].set_ylim(zmin,zmax+250); axes[1].legend(loc='upper right'); axes[1].set_title(f'Sección vertical central Y={mesh.cell_centers_y[y_ind]:.0f} m'); sm=mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0.0,vmax),cmap='viridis'); sm.set_array([]); fig.colorbar(sm,cax=cax,label='Magnetic susceptibility (SI)'); plt.show()

## Exportación y dictamen

In [ ]:
previous={'caso':'Anterior: contraste firmado','misfit_final':229.921,'rmse_validacion_nt':162.486,'correlacion_validacion':0.591,'chi_min_si':-0.098944,'chi_max_si':0.126952,'negativas_pct':53.933324,'lower_bound_pct':0.0,'upper_bound_pct':0.0}
current={'caso':'Nuevo: susceptibilidad no negativa','misfit_final':result['phi_d'],'rmse_validacion_nt':rmse_h,'correlacion_validacion':corr_h,'chi_min_si':m.min(),'chi_max_si':m.max(),'negativas_pct':physical_negative_pct,'lower_bound_pct':at_lower,'upper_bound_pct':at_upper}
display(pd.DataFrame([previous,current]).set_index('caso').round(6))
output=save_result(result,ROOT); print(f'Resultado guardado en: {output.relative_to(ROOT)}'); print(f'Modelo 3D NumPy: {output.with_name(output.stem + "_model_3d.npy").relative_to(ROOT)}'); print(f'Modelo 3D VTK: {output.with_name(output.stem + "_model_3d.vtk").relative_to(ROOT)}')

### Interpretación provisional

La convergencia del entrenamiento verifica consistencia numérica; la validación espacial prueba predicción fuera de muestra. Ninguna de las dos elimina la no unicidad. Antes de interpretar unidades geológicas se deben evaluar fecha/campo IGRF, remanencia, incertidumbre, profundidad y estabilidad frente a regularización.